In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl

# Argoverse uses as default 2s past and 3s future, both at 10HZ
OBS_LEN = 20
PRED_LEN = 30

In [4]:
#Lets make a CSV loader for the agents
def load_agent_xy(csv_path: Path):
    """Return timestamps and XY for the AGENT track in one scene."""
    df = pd.read_csv(csv_path).sort_values(["TIMESTAMP", "TRACK_ID"])
    agent_id = df.loc[df["OBJECT_TYPE"] == "AGENT", "TRACK_ID"].iloc[0]
    a = df[df["TRACK_ID"] == agent_id].sort_values("TIMESTAMP")
    ts = a["TIMESTAMP"].to_numpy()
    xy = a[["X", "Y"]].to_numpy(dtype=np.float32)
    return ts, xy

In [5]:
#Dataset, agent only. one observation and future per scene
class ArgoverseAgentOnlyDataset(Dataset):
    """
    Each item:
      past:  (OBS_LEN, 2)  positions relative to last observed point
      future:(PRED_LEN,2)  positions relative to last observed point (supervision)
      meta:  dict with scene path, agent_id, city, origin_xy
    """
    def __init__(self, files, obs_len=OBS_LEN, pred_len=PRED_LEN, drop_short=True):
        self.files = [Path(f) for f in files]
        self.obs_len = obs_len
        self.pred_len = pred_len
        self.drop_short = drop_short

        # pre-index scenes that have enough frames
        self.index = []
        for p in self.files:
            ts, xy = load_agent_xy(p)
            if len(xy) >= obs_len + pred_len:
                self.index.append((p, len(xy)))  # store length in case you later want sliding windows
            elif not drop_short:
                self.index.append((p, len(xy)))

        if len(self.index) == 0:
            raise RuntimeError("No scenes with enough frames. Check your paths or lengths.")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        p, n = self.index[i]
        ts, xy = load_agent_xy(p)

        # simplest choice: take the first obs_len as past, next pred_len as future
        # (Argoverse scenes are typically 50 frames so this lines up with the benchmark)
        past  = xy[:self.obs_len]
        fut   = xy[self.obs_len:self.obs_len + self.pred_len]

        # center everything at the last observed point (common trick)
        origin = past[-1]
        past_rel = past - origin
        fut_rel  = fut  - origin

        item = {
            "past":  torch.from_numpy(past_rel),   # (obs_len, 2)
            "future":torch.from_numpy(fut_rel),    # (pred_len, 2)
            "origin": torch.from_numpy(origin),    # (2,)
            "path":  str(p)
        }
        return item


In [8]:
#Create the lightning DataModule
class ArgoAgentDataModule(pl.LightningDataModule):
    def __init__(self, train_dir, val_dir=None, test_dir=None, batch_size=64, num_workers=0,
                 obs_len=OBS_LEN, pred_len=PRED_LEN):
        super().__init__()
        self.train_dir = Path(train_dir) if train_dir else None
        self.val_dir   = Path(val_dir)   if val_dir   else None
        self.test_dir  = Path(test_dir)  if test_dir  else None
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.obs_len = obs_len
        self.pred_len = pred_len

    def _collect_csvs(self, root: Path):
        if root is None: return []
        return sorted(root.glob("*.csv"))

    def setup(self, stage=None):
        if self.train_dir:
            train_files = self._collect_csvs(self.train_dir)
            self.train_ds = ArgoverseAgentOnlyDataset(train_files, self.obs_len, self.pred_len)
        if self.val_dir:
            val_files = self._collect_csvs(self.val_dir)
            self.val_ds = ArgoverseAgentOnlyDataset(val_files, self.obs_len, self.pred_len)
        if self.test_dir:
            test_files = self._collect_csvs(self.test_dir)
            self.test_ds = ArgoverseAgentOnlyDataset(test_files, self.obs_len, self.pred_len)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True,
                          num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        if hasattr(self, "val_ds"):
            return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False,
                              num_workers=self.num_workers, pin_memory=True)
        return None

    def test_dataloader(self):
        if hasattr(self, "test_ds"):
            return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False,
                              num_workers=self.num_workers, pin_memory=True)
        return None


In [ ]:
#Link to folders
# match your layout; change if you have val/test downloaded
TRAIN_DIR = "data/forecasting_train_v1.1/train/data/"
VAL_DIR   = "data/forecasting_val_v1.1/val/data/"
TEST_DIR  = "data/forecasting_test_v1.1/test/data/"

dm = ArgoAgentDataModule(train_dir=TRAIN_DIR, val_dir=VAL_DIR, test_dir=TEST_DIR,
                         batch_size=64, num_workers=0)
dm.setup()

batch = next(iter(dm.train_dataloader()))
print("past:", batch["past"].shape, "future:", batch["future"].shape)
print("example paths:", batch["path"][:3])
